# Phase 2D — Independent verification of Gemini Phase 2B/2C

Verification only. Recalculate saved predictions and catalog statistics; inspect source and recorded hashes. No model loading, training, optimization, data changes, or model selection. Test targets are used only to verify Phase 2C descriptive statistics. All audit code is here; original artifacts remain read-only.

PASS = reproduced; PARTIAL = supported with limits; FAIL = contradicted; UNKNOWN = insufficient evidence. Historical runtime behavior is not inferred solely from present-day source code.


In [1]:
from pathlib import Path
import ast, hashlib, json, math, platform, re
import numpy as np
import pandas as pd
import torch
import scipy
from scipy.stats import pearsonr, spearmanr, wasserstein_distance, ks_2samp
ROOT = Path.cwd()
if ROOT.name != "bilstm new updated":
    ROOT = ROOT / "bilstm new updated"
ROOT = ROOT.resolve()
DATA = ROOT / "dataset/v2_timestamp_fixed"
RES = ROOT / "phase2_bilstm_results"
HUB = RES / "huber_experiment"
BASE = RES / "variants/C"
DIAG = RES / "error_diagnosis"
OUT = RES / "gemini_verification"
OUT.mkdir(parents=True, exist_ok=True)
def readj(path):
    return json.loads(path.read_text(encoding="utf-8"))
def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()
def normalized(value):
    if isinstance(value, dict):
        return {str(k): normalized(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)):
        return [normalized(v) for v in value]
    if isinstance(value, np.ndarray):
        return normalized(value.tolist())
    if isinstance(value, np.generic):
        return normalized(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value
def savej(name, value):
    (OUT/name).write_text(json.dumps(normalized(value), indent=2, allow_nan=False), encoding="utf-8")
findings = []
def claim(section, claim_text, status, evidence, kind="verification"):
    findings.append(dict(section=section, claim=claim_text, status=status, kind=kind, evidence=str(evidence)))
def src(nb):
    return "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"]=="code")
nb_old = readj(ROOT/"phase2_bilstm_training.ipynb")
nb_hub = readj(ROOT/"phase2b_huber_loss_experiment.ipynb")
code_old, code_hub = src(nb_old), src(nb_hub)
tree_old, tree_hub = ast.parse(code_old), ast.parse(code_hub)
report_b = (HUB/"huber_experiment_report.md").read_text(encoding="utf-8")
report_c = (DIAG/"phase2c_feature_informativeness_report.md").read_text(encoding="utf-8")
gem_c = readj(DIAG/"phase2c_feature_audit.json")
gem_b = readj(HUB/"huber_delta2_metrics.json")
protocol = readj(RES/"protocol.json")
hyper = readj(RES/"best_hyperparameters.json")["parameters"]
scaling = readj(DATA/"catalog_train_scalers.json")
schema = readj(DATA/"feature_schema.json")
target_definition = readj(DATA/"target_definition.json")
histories = {"delta1":readj(BASE/"history.json"),"delta2":readj(HUB/"history.json")}
fits = {"delta1":readj(BASE/"fit_summary.json"),"delta2":readj(HUB/"fit_summary.json")}
registries = [
    (RES, RES/"artifact_hashes.json", "files"),
    (DATA, DATA/"catalog_artifact_hashes.json", "artifacts")
]
integrity = []
for parent, registry_path, key in registries:
    for relative, expected in readj(registry_path)[key].items():
        path = parent/relative.replace("\\","/")
        actual = sha(path) if path.exists() else None
        integrity.append(dict(path=str(path.relative_to(ROOT)), expected=expected,
                              actual=actual, match=(actual==expected)))
protected_paths = {ROOT/"phase2_bilstm_training.ipynb", ROOT/"phase2b_huber_loss_experiment.ipynb",
                   ROOT/"phase1b_imbalance_noise_preprocessing.ipynb",
                   DIAG/"phase2c_feature_audit.json", DIAG/"phase2c_feature_informativeness_report.md"}
# Only known audit directories and registered artifacts; no repository scan.
protected_paths.update(p for p in HUB.iterdir() if p.is_file())
protected_paths.update(BASE/name for name in ["history.json","fit_summary.json","validation_predictions.npz"])
protected_paths.add(RES/"optuna_study.sqlite3")
protected_paths.update(p for p in (HUB/"plots").glob("*.png"))
for name in ["feature_correlation_drift.png","temporal_distribution_shift.png"]:
    if (DIAG/name).exists():
        protected_paths.add(DIAG/name)
for parent, registry_path, key in registries:
    protected_paths.add(registry_path)
    protected_paths.update(parent/rel.replace("\\","/") for rel in readj(registry_path)[key])
snapshot = {str(p.relative_to(ROOT)):sha(p) for p in sorted(protected_paths) if p.is_file()}
claim("Integrity","Registered original Phase 2 and corrected v2 artifacts",
      "PASS" if all(x["match"] for x in integrity) else "FAIL",
      f'{sum(x["match"] for x in integrity)}/{len(integrity)} hashes match existing registries. Checkpoints read as bytes only.')
dataset = torch.load(DATA/"bilstm_catalog_seq20.pt", map_location="cpu", weights_only=True)
X = {s:dataset["X_"+s].numpy().astype(np.float64) for s in ["train","val","test"]}
y = {s:dataset["y_reg_"+s].numpy().astype(np.float64).reshape(-1) for s in X}
del dataset
manifest = pd.read_csv(DATA/"split_manifest.csv")
catalog = pd.read_csv(DATA/"catalog_events.csv", dtype={"source_id":str})
catalog["time"] = pd.to_datetime(catalog["time"], utc=True)
parts = {s:manifest[manifest.split==s].reset_index(drop=True) for s in X}
features = schema["ordered_features"]
last = {s:X[s][:,-1,:]*np.array(scaling["scaler"]["scale"])+np.array(scaling["scaler"]["mean"]) for s in X}
audit = {"audit_date":"2026-11-10","scope":"verification only; no model load/inference/training",
         "environment":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
                        "scipy":scipy.__version__,"torch":torch.__version__},
         "registered_integrity":integrity, "source_hashes_before":snapshot}
print("Loaded corrected v2 once; registered artifact matches:",sum(x["match"] for x in integrity),"/",len(integrity))


Loaded corrected v2 once; registered artifact matches: 29 / 29


In [2]:
# Independent metric formulas; no invocation of Gemini's evaluation functions.
predictions = {}
for label,folder in [("delta1",BASE),("delta2",HUB)]:
    with np.load(folder/"validation_predictions.npz", allow_pickle=False) as z:
        actual = np.asarray(z["actual"],dtype=np.float64).reshape(-1)
        prediction = np.asarray(z["prediction"],dtype=np.float64).reshape(-1)
    assert np.array_equal(actual, y["val"]), label+" target order mismatch"
    assert prediction.shape==actual.shape and np.isfinite(prediction).all()
    predictions[label] = prediction
def independent_metrics(actual,prediction,delta):
    err = prediction-actual
    absolute = np.abs(err)
    high = actual>=4.5
    low = actual<3
    return dict(n=len(actual),mae=float(absolute.mean()),rmse=float(np.sqrt((err**2).mean())),
        r2=float(1-(err**2).sum()/((actual-actual.mean())**2).sum()),
        huber=float(np.where(absolute<=delta,0.5*err**2,delta*(absolute-0.5*delta)).mean()),
        pred_mean=float(prediction.mean()),pred_std=float(prediction.std(ddof=0)),
        actual_mean=float(actual.mean()),actual_std=float(actual.std(ddof=0)),
        pred_min=float(prediction.min()),pred_max=float(prediction.max()),
        high_count=int(high.sum()),high_mae=float(absolute[high].mean()),
        high_underprediction_rate=float((err[high]<0).mean()),high_bias=float(err[high].mean()),
        low_count=int(low.sum()),low_mae=float(absolute[low].mean()),
        predicted_actual_sd_ratio=float(prediction.std()/actual.std()),
        predicted_actual_variance_ratio=float(prediction.var()/actual.var()),
        mean_distance_from_actual=float(abs(prediction.mean()-actual.mean())))
metrics_b = {k:independent_metrics(y["val"],p,float(k[-1])) for k,p in predictions.items()}
checks_b = []
for label,gemkey in [("delta1","delta1_baseline"),("delta2","delta2_experiment")]:
    for field,reported in gem_b[gemkey].items():
        if field in metrics_b[label]:
            got = metrics_b[label][field]
            checks_b.append(dict(configuration=label,field=field,reported=reported,recomputed=got,
                                 match=bool(np.isclose(got,reported,rtol=1e-10,atol=1e-10))))
a,b = metrics_b["delta1"],metrics_b["delta2"]
changes = dict(mae_delta2_minus_delta1=b["mae"]-a["mae"],
    mae_improvement_percent=100*(a["mae"]-b["mae"])/a["mae"],
    prediction_sd_reduction_percent=100*(a["pred_std"]-b["pred_std"])/a["pred_std"],
    high_mae_change=b["high_mae"]-a["high_mae"],
    paired_error_wins=int((np.abs(predictions["delta2"]-y["val"])<np.abs(predictions["delta1"]-y["val"])).sum()),
    paired_error_losses=int((np.abs(predictions["delta2"]-y["val"])>np.abs(predictions["delta1"]-y["val"])).sum()))
# Descriptive paired errors only; do not introduce a new significance experiment.
paired = np.abs(predictions["delta2"]-y["val"])-np.abs(predictions["delta1"]-y["val"])
changes["paired_error_difference_sd"] = float(paired.std())
changes["paired_error_difference_quantiles"] = np.quantile(paired,[0,.25,.5,.75,1]).tolist()
best_rows = {}
for label,hist in histories.items():
    best_row = min(hist,key=lambda h:h["val_mae"])
    best_rows[label] = best_row
    assert best_row["epoch"]==fits[label]["best_epoch"]
    assert np.isclose(best_row["val_mae"],metrics_b[label]["mae"],atol=1e-10,rtol=0)
    metrics_b[label]["clean_train_mae_logged"] = best_row["train_mae"]
    metrics_b[label]["gap_val_minus_train_mae_logged"] = metrics_b[label]["mae"]-best_row["train_mae"]
    metrics_b[label]["best_epoch"] = best_row["epoch"]
    metrics_b[label]["epochs_run"] = len(hist)
    metrics_b[label]["training_seconds_logged"] = fits[label]["training_seconds"]
claim("Phase 2B","Saved validation metrics and target alignment",
      "PASS" if all(v["match"] for v in checks_b) else "FAIL",
      f'{len(checks_b)} JSON metric comparisons; both actual arrays exactly equal v2 validation labels in manifest order.')
claim("Phase 2B","Reported 20.7% SD reduction and approximately 0.55% MAE gain","PASS",
      f'SD reduction {changes["prediction_sd_reduction_percent"]:.6f}%; MAE improvement {changes["mae_improvement_percent"]:.6f}%.')
claim("Phase 2B","Report delta2 validation Huber loss 0.2312","FAIL",
      f'Independent delta2 Huber = {b["huber"]:.10f}; JSON agrees, prose does not.',"numerical")
claim("Phase 2B","Prediction mean moves closer to validation center","FAIL",
      f'Absolute mean offset increases {a["mean_distance_from_actual"]:.6f} -> {b["mean_distance_from_actual"]:.6f}.',"interpretation")
claim("Phase 2B","19.8% of validation variance is retained","FAIL",
      f'SD ratio {100*b["predicted_actual_sd_ratio"]:.3f}%; variance ratio {100*b["predicted_actual_variance_ratio"]:.3f}%.',"numerical")
claim("Phase 2B","Report delta2 training duration 10.3 seconds","FAIL",
      f'fit_summary/history elapsed = {fits["delta2"]["training_seconds"]:.9f} seconds; actual wall-clock duration is not independently logged by this audit.',"numerical")
claim("Phase 2B","Smaller MAE generalization gap proves less overfitting","PARTIAL",
      f'Logged gap {metrics_b["delta1"]["gap_val_minus_train_mae_logged"]:.6f} -> {metrics_b["delta2"]["gap_val_minus_train_mae_logged"]:.6f}; training MAE worsens and compression increases. Different distributions and one seed prevent that causal inference.',"interpretation")
claim("Phase 2B","Statistically reliable improvement established","UNKNOWN",
      f'Paired MAE change {changes["mae_delta2_minus_delta1"]:.6f}; same validation used for epoch selection, overlapping windows, 75 high targets and one seed. No dependence-aware confidence intervals or repeated-seed evidence.')
# The baseline history lacks val_pred_std, so the conditional plot uses the RMSE fallback.
plot_uses_rmse = not any("val_pred_std" in h for h in histories["delta1"])
claim("Phase 2B","Saved RMSE plot chooses RMSE from the current baseline history",
      "PASS" if plot_uses_rmse else "FAIL",
      "Notebook cell 9 conditionally prefers SD, but baseline val_pred_std is absent; current executed fallback is RMSE. Fragile code, not a demonstrated error in this figure.")
audit["phase2b"] = dict(metrics=metrics_b,changes=changes,json_metric_checks=checks_b,
                       best_history_rows=best_rows,rmse_plot_current_fallback_correct=plot_uses_rmse)
print("Validation MAE:",round(a["mae"],6),"->",round(b["mae"],6),
      "; SD reduction:",round(changes["prediction_sd_reduction_percent"],3),"%")


Validation MAE: 0.4724 -> 0.469827 ; SD reduction: 20.668 %


In [3]:
# Static source + recorded execution evidence. Never execute training code or load model weights.
def definition(tree,name):
    return next((n for n in tree.body if isinstance(n,(ast.ClassDef,ast.FunctionDef)) and n.name==name),None)
def dump_definition(tree,name):
    node = definition(tree,name)
    return ast.dump(node,include_attributes=False) if node is not None else None
def constant_assignments(tree):
    values={}
    for n in tree.body:
        if isinstance(n,ast.Assign):
            try:
                val=ast.literal_eval(n.value)
            except (ValueError,TypeError):
                continue
            for t in n.targets:
                if isinstance(t,ast.Name):
                    values[t.id]=val
    return values
const_old,const_new = constant_assignments(tree_old),constant_assignments(tree_hub)
same_arch = dump_definition(tree_old,"BiLSTMRegressor")==dump_definition(tree_hub,"BiLSTMRegressor")
same_init = fits["delta1"]["initialization_sha256"]==fits["delta2"]["initialization_sha256"]
parent_checks = [
    dict(epoch=h2["epoch"],
         match=all(h1[k]==h2[k] for k in ["parent_indices_sha256","unique_parents","repeated_draws","effective_high_fraction"]))
    for h1,h2 in zip(histories["delta1"],histories["delta2"])
]
sampling_nb = readj(ROOT/"phase1b_imbalance_noise_preprocessing.ipynb")
sampling_tree = ast.parse(src(sampling_nb))
consumer_names={"draw_parent_indices","coordinate_noise","diagnostic_variant","prepare_training_epoch"}
consumer_nodes=[n for n in sampling_tree.body if isinstance(n,ast.FunctionDef) and n.name in consumer_names]
consumer_source=ast.unparse(ast.Module(body=consumer_nodes,type_ignores=[]))
consumer_sha=hashlib.sha256(consumer_source.encode()).hexdigest()
calls_new = [ast.unparse(n) for n in ast.walk(tree_hub) if isinstance(n,ast.Call)]
subscripts_new = sorted(set(ast.unparse(n) for n in ast.walk(tree_hub) if isinstance(n,ast.Subscript)
                            and isinstance(n.value,ast.Name) and n.value.id=="mapped"))
test_literals = sorted(set(n.value for n in ast.walk(tree_hub) if isinstance(n,ast.Constant)
                          and isinstance(n.value,str) and "test" in n.value.lower()))
source_settings = {k:const_new.get(k) for k in ["SEED","MAX_EPOCHS","PATIENCE","HUBER_DELTA","GRAD_CLIP"]}
controls = dict(
    baseline_protocol=protocol,baseline_hyperparameters=hyper,
    baseline_fit_hyperparameters=fits["delta1"]["fit_key"]["hyperparameters"],
    delta2_module_constants=source_settings,architecture_ast_equal=same_arch,
    initial_weights_hash_equal=same_init,initialization_sha256=fits["delta2"]["initialization_sha256"],
    per_epoch_sampling=parent_checks,approved_consumer_sha256=consumer_sha,
    approved_consumer_matches_original_protocol=(consumer_sha==protocol["consumer_source_sha256"]),
    mapped_tensor_keys_in_delta2_source=subscripts_new,test_strings_in_delta2_source=test_literals,
    optimizer_calls=[c for c in calls_new if c.startswith("torch.optim.AdamW(")],
    loss_calls=[c for c in calls_new if c.startswith("nn.HuberLoss(")],
    clipping_calls=[c for c in calls_new if "clip_grad_norm_(" in c],
    selection_expression=[ast.unparse(n.value) for n in ast.walk(tree_hub) if isinstance(n,ast.Assign)
                          and any(isinstance(t,ast.Name) and t.id=="improved" for t in n.targets)],
    hyperparameter_source_lines=[line.strip() for line in code_hub.splitlines()
                                if "best_cfg =" in line or "best_hyperparameters.json" in line],
    recorded_checkpoint_hashes={label:dict(recorded=fits[label]["best_checkpoint_sha256"],
        actual=sha(folder/"best.pt")) for label,folder in [("delta1",BASE),("delta2",HUB)]},
    executed_cells=[dict(index=i,execution_count=c.get("execution_count"),
                        errors=[o.get("ename") for o in c.get("outputs",[]) if o.get("output_type")=="error"])
                    for i,c in enumerate(nb_hub["cells"]) if c["cell_type"]=="code"]
)
identity = scaling["target_transform"]=={"name":"identity","offset":0.0,"scale":1.0}
assert identity and protocol["huber_delta"]==1 and const_new["HUBER_DELTA"]==2
claim("Phase 2B controls","Original delta and original ML units","PASS",
      "Registered protocol: delta1=1; delta2 notebook constant=2; identity target transform. No target conversion.")
claim("Phase 2B controls","Matching architecture, initialization and sampled parents",
      "PASS" if same_arch and same_init and all(x["match"] for x in parent_checks) else "FAIL",
      f'Architecture AST equality={same_arch}; initialization hash equality={same_init}; all {len(parent_checks)} common epochs have matching parent/order hashes and sample counts.')
claim("Phase 2B controls","Existing full-precision hyperparameters, AdamW and sampler source","PASS",
      f'New source reads registered best_hyperparameters.json, AdamW(model.parameters()), same clipping/seed/early stopping settings; consumer SHA matches original={controls["approved_consumer_matches_original_protocol"]}. Controls JSON retains exact values.')
claim("Phase 2B controls","Exact historical one-variable training-control equivalence","UNKNOWN",
      "Source and recorded initialization/sampling strongly support equivalence, but no immutable delta2 run configuration/environment/consumer snapshot establishes every runtime setting. Current source cannot establish full historical execution equivalence.")
claim("Phase 2B controls","Validation-only selection in supplied delta2 notebook","PASS",
      "Source indexes X_train,y_reg_train,y_binary_train,X_val,y_reg_val only; minimum validation MAE controls checkpoint. No test target indexing or test predictions.")
claim("Phase 2B controls","No test-label access anywhere during historical experiment","UNKNOWN",
      "No external execution/access log. The monolithic .pt contains all splits; hashing file bytes and memory mapping are not evidence of test-label use. Present source supports intended protocol, not an absolute historical guarantee.")
best_match=all(v["recorded"]==v["actual"] for v in controls["recorded_checkpoint_hashes"].values())
claim("Phase 2B controls","Best checkpoints correspond to validation-selected epochs","PARTIAL",
      f'Both checkpoint hashes match fit summaries={best_match}; history minima and saved prediction MAE match epochs12/11. Source saves on val-MAE improvements and reloads best before prediction. Checkpoint metadata/weights not deserialized per verification scope.')
claim("Integrity","Phase 2B outputs separate from original Phase 2","PASS",
      "Dedicated huber_experiment directory; original registered artifacts unchanged. Not an immutable resumable run: rerunning notebook would overwrite delta2 files.")
phase2c_notebook = ROOT/"phase2c_feature_temporal_audit.ipynb"
claim("Phase 2C provenance","Original executable Phase 2C notebook","UNKNOWN" if not phase2c_notebook.exists() else "PASS",
      "Named notebook absent from scoped project filename search; report, JSON and two figures exist. Reconstructed calculations cannot replace missing execution provenance.")
claim("Integrity","Phase 2C performed no training or model modifications","PARTIAL",
      "Registered original checkpoints unchanged; no Phase 2C executable source/run log to establish that no other training occurred.")
audit["training_controls"] = controls
print("Recorded controls: architecture",same_arch,"initialization",same_init,
      "; matching sampler epochs",sum(x["match"] for x in parent_checks))


Recorded controls: architecture True initialization True ; matching sampler epochs 19


In [4]:
# Alignment and independent final-step statistics, using retained catalog IDs and UTC timestamps.
cat_by_id = catalog.set_index("source_id")
assert cat_by_id.index.is_unique
assert catalog["event_index"].is_monotonic_increasing and catalog["time"].is_monotonic_increasing
alignment={}
last_indices={}
for s,df in parts.items():
    input_ids=df.input_event_ids.map(json.loads)
    last_ids=input_ids.map(lambda ids:str(ids[-1]))
    ti=pd.to_datetime(df.target_timestamp_utc,utc=True)
    input_times=df.input_timestamps_utc.map(json.loads)
    strict=all(all(pd.Timestamp(t)<target for t in times) for times,target in zip(input_times,ti))
    target_ids=df.target_event_id.astype(str)
    source_targets=cat_by_id.loc[target_ids,"magnitude"].to_numpy()
    last_indices[s]=cat_by_id.loc[last_ids,"event_index"].to_numpy(dtype=int)
    source_times=cat_by_id.loc[last_ids,"time"].reset_index(drop=True)
    gaps=(ti-source_times).dt.total_seconds().to_numpy()
    expected_y=source_targets.astype(np.float32).astype(np.float64)
    alignment[s]=dict(shape=list(X[s].shape),n=len(df),unique_target_ids=target_ids.nunique(),
        target_values_match_source=bool(np.array_equal(expected_y,y[s])),
        target_values_match_manifest=bool(np.array_equal(df.target_magnitude.to_numpy().astype(np.float32).astype(np.float64),y[s])),
        final_input_magnitude_matches=bool(np.allclose(last[s][:,0],cat_by_id.loc[last_ids,"magnitude"].to_numpy(),atol=1e-6,rtol=0)),
        input_before_target=bool(strict),target_times_sorted=bool(ti.is_monotonic_increasing),
        target_gap_matches_manifest=bool(np.allclose(gaps,df.input_to_target_gap_seconds.to_numpy(),atol=1e-6,rtol=0)),
        first_target_time=ti.iloc[0].isoformat(),last_target_time=ti.iloc[-1].isoformat(),
        target_gap_mean_days=float(gaps.mean()/86400),target_gap_median_days=float(np.median(gaps)/86400))
assert all(all(v[k] for k in ["target_values_match_source","target_values_match_manifest","final_input_magnitude_matches","input_before_target","target_times_sorted","target_gap_matches_manifest"]) for v in alignment.values())
assert manifest.target_event_id.is_unique
assert pd.to_datetime(manifest.target_timestamp_utc,utc=True).is_monotonic_increasing
claim("Phase 2C","Target alignment, shapes and chronological partitions","PASS",
      "4973/1065/1067 samples, 20x10; manifest labels match source and tensors; unique target IDs; all input timestamps strictly before targets; exact target-gap checks pass.")
half=len(y["train"])//2
periods={}
for name,start,stop in [("train_h1",0,half),("train_h2",half,len(y["train"]))]:
    times=pd.to_datetime(parts["train"].iloc[start:stop].target_timestamp_utc,utc=True)
    periods[name]=dict(row_start_zero_based=start,row_stop_exclusive=stop,n=stop-start,
                      first_target=times.iloc[0].isoformat(),last_target=times.iloc[-1].isoformat())
rows=[]
for i,feat in enumerate(features):
    assert i==feat["index"] and gem_c["feature_metrics"][i]["name"]==feat["name"]
    row=dict(index=i,name=feat["name"],units=feat["units"])
    for s,prefix in [("train","tr"),("val","val")]:
        f=last[s][:,i]
        row.update({prefix+"_mean":float(f.mean()),prefix+"_std":float(f.std(ddof=0)),
                    prefix+"_median":float(np.median(f)),prefix+"_iqr":float(np.quantile(f,.75)-np.quantile(f,.25)),
                    prefix+"_min":float(f.min()),prefix+"_max":float(f.max())})
    for key,xx,yy in [("train",last["train"][:,i],y["train"]),
                       ("h1",last["train"][:half,i],y["train"][:half]),
                       ("h2",last["train"][half:,i],y["train"][half:]),
                       ("val",last["val"][:,i],y["val"])]:
        r,p=pearsonr(xx,yy);rho,prho=spearmanr(xx,yy)
        row["r_"+key]=float(r);row["rho_"+key]=float(rho)
        row["p_r_"+key]=float(p);row["p_rho_"+key]=float(prho)
    row["delta_r"]=row["r_val"]-row["r_train"]
    row["delta_rho"]=row["rho_val"]-row["rho_train"]
    for s,prefix in [("train","tr"),("val","val")]:
        xx=last[s][:,i];hi=xx[y[s]>=4.5];lo=xx[y[s]<4.5]
        row[prefix+"_mean_high_target"]=float(hi.mean())
        row[prefix+"_mean_low_target"]=float(lo.mean())
        row["d_"+s]=float((hi.mean()-lo.mean())/xx.std(ddof=0))
        pooled=np.sqrt(((len(hi)-1)*hi.var(ddof=1)+(len(lo)-1)*lo.var(ddof=1))/(len(hi)+len(lo)-2))
        row["cohens_d_pooled_"+s]=float((hi.mean()-lo.mean())/pooled)
    row["val_oob_pct"]=float(100*np.mean((last["val"][:,i]<last["train"][:,i].min())|(last["val"][:,i]>last["train"][:,i].max())))
    rows.append(row)
feature_checks=[]
for got,reported in zip(rows,gem_c["feature_metrics"]):
    for field,value in reported.items():
        if isinstance(value,(int,float)):
            feature_checks.append(dict(index=got["index"],field=field,reported=value,recomputed=got[field],
                                       abs_difference=abs(got[field]-value),
                                       match=bool(np.isclose(got[field],value,rtol=1e-7,atol=1e-8))))
feature_frame=pd.DataFrame(rows)
corrmatrix=np.corrcoef(last["train"],rowvar=False)
redundancies=[dict(feature_1=features[i]["name"],feature_2=features[j]["name"],pearson_r=float(corrmatrix[i,j]))
              for i in range(10) for j in range(i+1,10) if abs(corrmatrix[i,j])>=.6]
claim("Phase 2C","All ten final-step feature JSON statistics/correlations",
      "PASS" if all(c["match"] for c in feature_checks) else "FAIL",
      f'{sum(c["match"] for c in feature_checks)}/{len(feature_checks)} numeric fields reproduced; inverse-scaled step19, population SD. See feature_verification.csv and metrics JSON.')
claim("Phase 2C","Exact early/late original method provenance","PARTIAL",
      f'First {half} and remaining {len(y["train"])-half} train rows reproduce H1/H2 correlations. Exact reconstructed UTC boundaries recorded. Original notebook/method log unavailable; matching reconstruction is not recovered source.')
claim("Phase 2C","Cohen d label in JSON/report","FAIL",
      f'Reported d divides group mean difference by full-split SD, not pooled within-group SD. Feature0 train: saved {rows[0]["d_train"]:.6f}, conventional pooled d {rows[0]["cohens_d_pooled_train"]:.6f}.',"interpretation")
claim("Phase 2C","Depth feature1 redundant with mean-depth8 at r=0.66","FAIL",
      f'Actual r(1,8)={corrmatrix[1,8]:.6f}; r(8,9)={corrmatrix[8,9]:.6f}. JSON correctly identifies pair8,9; report mislabels it.',"numerical")
claim("Phase 2C","No validation feature has abs(Pearson r)>0.20","FAIL",
      f'mean_magnitude_20 r={rows[6]["r_val"]:.9f}; Spearman={rows[6]["rho_val"]:.9f}. Boundary overstatement does not establish predictive usefulness.',"numerical")
claim("Phase 2C","Correlation weakening/sign changes indicate descriptive instability","PASS",
      f'Rolling-mean r {rows[6]["r_h1"]:.6f}/{rows[6]["r_h2"]:.6f}/{rows[6]["r_val"]:.6f}; overlapping histories and time trends prevent iid significance or causal inference.')
claim("Phase 2C","Univariate r proves absence of useful predictive signal or joint explained variance","FAIL",
      "Individual correlations omit nonlinear/joint/sequence information. Squared r for a single in-sample linear association is not joint or held-out model explained variance.", "interpretation")
audit["phase2c_features"]=dict(alignment=alignment,reconstructed_periods=periods,
    reconstruction="inverse-scaled final step19; H1 first floor(n_train/2) rows, H2 remainder; not recovered original code",
    statistics=rows,json_comparisons=feature_checks,redundant_pairs=redundancies)
print("Feature JSON reproduction:",sum(c["match"] for c in feature_checks),"/",len(feature_checks))


Feature JSON reproduction: 340 / 340


In [5]:
# Independently verify report table values, UTC gaps, rate denominators, and shift distances.
table_checks=[]
for line in report_c.splitlines():
    match=re.match(r"\|\s*\*\*(\d+)\*\*\s*\|",line)
    if not match:
        continue
    i=int(match.group(1));columns=line.split("|")
    for prefix,col in [("tr",4),("val",5)]:
        tokens=re.findall(r"[-+]?\d+(?:\.\d+)?",columns[col])
        if len(tokens)!=2:
            raise ValueError("Unexpected feature summary format")
        for field,token in zip(["mean","std"],tokens):
            reported=float(token)
            decimals=len(token.split(".")[1]) if "." in token else 0
            tolerance=.5*10**(-decimals)+1e-9
            got=rows[i][prefix+"_"+field]
            item=dict(index=i,name=features[i]["name"],field=prefix+"_"+field,
                      reported=reported,recomputed=got,rounding_tolerance=tolerance,
                      match=bool(abs(got-reported)<=tolerance))
            table_checks.append(item)
            if not item["match"]:
                claim("Phase 2C table",f'{features[i]["name"]}: {prefix} {field}',"FAIL",
                      f'Report {reported}; final-step recomputation and JSON {got:.8f} (outside displayed rounding tolerance).',"numerical")
# Explicit nanoseconds -> seconds/days; times are catalog origin times, not publication times.
times_ns=catalog.time.dt.as_unit("ns").astype("int64").to_numpy()
gaps_days=np.r_[np.nan,np.diff(times_ns)/1e9/86400]
left=np.searchsorted(times_ns,times_ns-int(30*86400*1e9),side="right")
event_rates=(np.arange(len(catalog))-left+1)/30.0
rate_gap_checks={}
summaries={}
for s in X:
    idx=last_indices[s]
    rate_error=float(np.max(np.abs(last[s][:,5]-event_rates[idx])))
    gap_error=float(np.max(np.abs(last[s][:,4]-gaps_days[idx])))
    rate_gap_checks[s]=dict(max_abs_event_rate_error=rate_error,max_abs_gap_days_error=gap_error,
                            matches=bool(rate_error<2e-6 and gap_error<1e-4))
    ts=pd.to_datetime(parts[s].target_timestamp_utc,utc=True)
    elapsed_days=(ts.iloc[-1]-ts.iloc[0]).total_seconds()/86400
    years=elapsed_days/365.2425
    yy=y[s]
    summaries[s]=dict(n=len(yy),target_mean=float(yy.mean()),target_sd=float(yy.std()),
        high_count=int((yy>=4.5).sum()),high_percent=float(100*(yy>=4.5).mean()),
        first_target=ts.iloc[0].isoformat(),last_target=ts.iloc[-1].isoformat(),
        elapsed_days=elapsed_days,elapsed_years_365_2425=years,
        targets_per_elapsed_year_N_over_span=len(yy)/years,
        interarrival_rate_per_year_Nminus1_over_span=(len(yy)-1)/years,
        last_input_rate_30d_mean=float(last[s][:,5].mean()),last_input_rate_30d_sd=float(last[s][:,5].std()),
        last_input_inter_event_days_mean=float(last[s][:,4].mean()),
        last_input_inter_event_days_median=float(np.median(last[s][:,4])),
        last_input_inter_event_days_sd=float(last[s][:,4].std()),
        mean_final_input_lat=float(last[s][:,2].mean()),mean_final_input_lon=float(last[s][:,3].mean()))
for name,period in periods.items():
    duration=(pd.Timestamp(period["last_target"])-pd.Timestamp(period["first_target"])).total_seconds()/86400/365.2425
    period["elapsed_years_365_2425"]=duration
    period["targets_per_elapsed_year_N_over_span"]=period["n"]/duration
distances={}
for s in ["val","test"]:
    ks=ks_2samp(y["train"],y[s],alternative="two-sided",method="auto")
    distances[s]=dict(wasserstein=float(wasserstein_distance(y["train"],y[s])),
                      ks_statistic=float(ks.statistic),ks_nominal_pvalue=float(ks.pvalue),
                      pvalue_interpretation="iid reference only; temporal dependence, discretized magnitudes and overlapping histories invalidate an unqualified inferential claim")
claim("Phase 2C","Time features use correct day units and 30-day boundaries",
      "PASS" if all(v["matches"] for v in rate_gap_checks.values()) else "FAIL",
      "Recomputed final-input gaps from UTC nanoseconds and rates in (time-30days,time], count/30; float32 reconstruction tolerance only. Target gaps separately match manifest.")
claim("Phase 2C","Mean target magnitudes and high-magnitude proportions","PASS",
      "; ".join(f'{s}: mean={v["target_mean"]:.6f}, high={v["high_count"]}/{v["n"]} ({v["high_percent"]:.4f}%)' for s,v in summaries.items()))
distance_claims=[
    ("train-val Wasserstein",.5247,distances["val"]["wasserstein"]),
    ("train-test Wasserstein",.9326,distances["test"]["wasserstein"]),
    ("train-val KS statistic",.3145,distances["val"]["ks_statistic"])]
for name,reported,got in distance_claims:
    claim("Phase 2C",name+" as printed","PASS" if abs(reported-got)<=.00005 else "FAIL",
          f'Report {reported:.4f}; independent {got:.10f}.',"numerical")
claim("Phase 2C","KS p<1e-15 as nominal iid calculation",
      "PASS" if distances["val"]["ks_nominal_pvalue"]<1e-15 else "FAIL",
      f'SciPy nominal p={distances["val"]["ks_nominal_pvalue"]:.5g}. It is not a dependence-robust significance assessment.')
claim("Phase 2C","KS/Pearson/Spearman significance valid without temporal-dependence qualifications","PARTIAL",
      "Distances and correlations are descriptive. Their conventional p values use independence assumptions; overlapping histories, catalog time trends and discretized/tied ML observations are not addressed. p=0 in saved JSON is numerical underflow, not literal probability zero.","interpretation")
rates_report={"train":50.,"val":942.,"test":1592.}
for s,printed in rates_report.items():
    got=summaries[s]["targets_per_elapsed_year_N_over_span"]
    claim("Phase 2C",f'{s} events/year denominator',"PASS" if abs(got-printed)<1 else "PARTIAL",
          f'Report ~{printed:.0f}; exact endpoint-span N/year={got:.3f}, N-1/year={summaries[s]["interarrival_rate_per_year_Nminus1_over_span"]:.3f}, span={summaries[s]["elapsed_years_365_2425"]:.6f} years. Rounded partial-year denominators explain discrepancies; neither rate establishes full catalog coverage.',"numerical")
claim("Phase 2C","Event-frequency measures establish physical seismicity or detection-rate changes","UNKNOWN",
      "These are counts/rates of retained ML target samples, not all source magnitudes or all regional earthquakes. Endpoint spans respect partial years but catalog uptime, missing periods, reporting-policy changes and true exposure are unavailable.","interpretation")
# Only metadata retained by corrected v2; no source CSV reload, no re-filtering or model experiment.
early=catalog[catalog.time < pd.Timestamp("2000-01-01",tz="UTC")]
low_early=early[early.magnitude<4]
early_evidence=dict(period="retained catalog before 2000-01-01 UTC",n=len(early),
                   count_below_4=len(low_early),
                   earliest_below_4=low_early[["source_id","time","magnitude"]].head(3).astype(str).to_dict("records"))
claim("Science","Mc approximately4 historically and1.5 now established","UNKNOWN",
      "No validated Mc estimation procedure, fit diagnostics, uncertainty or network-detection metadata supplied. Minimum observed magnitude is not Mc.","interpretation")
claim("Science","Changing catalog completeness is established cause of apparent predictability","FAIL",
      "Temporal associations and retained-event distribution changes are observed; a causal completeness explanation and claimed learned pseudo-rule were not tested. Detection/reporting changes, magnitude selection, spatial sampling and physical clustering remain alternative explanations.","interpretation")
claim("Science","Gutenberg-Richter implies independence of successive magnitudes","FAIL",
      "A marginal frequency-magnitude relationship specifies neither joint temporal dependence nor conditional independence. No serial/conditional-independence analysis is supplied.","interpretation")
claim("Science","Modern validation catalog is stationary and contains no predictive information","UNKNOWN",
      "No stationarity test or validated conditional/multivariate predictive-signal analysis is supplied. Small univariate correlations do not establish either claim.","interpretation")
claim("Science","Huber change has a proven mean-collapse mechanism","FAIL",
      "Compression increased for this recorded fit; Huber is optimized on weighted training samples. Validation majority is not the gradient objective. Huber optimum is generally neither exactly a mean nor necessarily variance-collapsed.","interpretation")
claim("Science","Network deployment and geographic centroid explain the shift","PARTIAL",
      "Centroid and distribution changes can be reproduced; no station/network coverage history establishes a network-deployment cause.","interpretation")
audit["phase2c_temporal"]=dict(split_summaries=summaries,distances=distances,
    rate_gap_checks=rate_gap_checks,feature_report_table_checks=table_checks,
    early_catalog_evidence=early_evidence,
    rate_definition="N/(last target origin time-first target origin time); 365.2425 days/year. N-1 interarrival rate supplied separately. Not verified catalog exposure.",
    mean_final_step_rate_test_train_ratio=summaries["test"]["last_input_rate_30d_mean"]/summaries["train"]["last_input_rate_30d_mean"])
print("Independent distances: train-val W",round(distances["val"]["wasserstein"],6),
      "KS",round(distances["val"]["ks_statistic"],6),
      "; feature report table mismatches",sum(not x["match"] for x in table_checks))


Independent distances: train-val W 0.524702 KS 0.314511 ; feature report table mismatches 19


In [6]:
# Verify saved notebook/CSV outputs, then write the independent report without altering sources.
csv_b=pd.read_csv(HUB/"huber_loss_comparison.csv")
csv_metric_map={"Validation MAE":"mae","Validation RMSE":"rmse","Validation R2":"r2",
                "ML>=4.5 Validation MAE":"high_mae","ML>=4.5 Signed Bias":"high_bias",
                "Prediction Mean":"pred_mean","Prediction Std Dev":"pred_std"}
csv_checks=[]
for display,field in csv_metric_map.items():
    row=csv_b[csv_b.Metric==display].iloc[0]
    for label,column in [("delta1","Baseline (delta=1)"),("delta2","Experiment (delta=2)")]:
        reported=float(row[column]);got=metrics_b[label][field]
        csv_checks.append(dict(metric=display,configuration=label,reported=reported,recomputed=got,
                               match=bool(abs(reported-got)<=.000050001)))
execution_text="\n".join(
    "".join(o.get("text",[])) if isinstance(o.get("text",""),list) else o.get("text","")
    for c in nb_hub["cells"] for o in c.get("outputs",[]) if o.get("output_type")=="stream")
expected_display_values=[f'{metrics_b[label][field]:.4f}' for label in ["delta1","delta2"]
                         for field in ["mae","rmse","r2","pred_std","high_mae"]]
output_values_present=all(v in execution_text for v in expected_display_values)
claim("Phase 2B","Saved comparison CSV and notebook metric output","PASS" if all(c["match"] for c in csv_checks) and output_values_present else "FAIL",
      f'{sum(c["match"] for c in csv_checks)}/{len(csv_checks)} rounded metric cells match; key four-decimal values present in saved notebook output={output_values_present}. Markdown discrepancies are listed separately.')
audit["phase2b"]["csv_checks"]=csv_checks
audit["phase2b"]["saved_notebook_output_contains_recomputed_values"]=output_values_present
assert controls["approved_consumer_matches_original_protocol"]
assert const_new["SEED"]==42 and const_new["MAX_EPOCHS"]==50 and const_new["PATIENCE"]==8
assert fits["delta1"]["fit_key"]["hyperparameters"]==hyper
assert all(v["recorded"]==v["actual"] for v in controls["recorded_checkpoint_hashes"].values())
# Verify source preservation against this audit's starting snapshot.
after={rel:sha(ROOT/rel) for rel in snapshot}
preserved=all(after[rel]==snapshot[rel] for rel in snapshot)
assert preserved,"Source artifact changed while verification ran"
claim("Integrity","Gemini/Codex source artifacts unchanged by this verification","PASS",
      f'{len(snapshot)} inspected source files have identical before/after SHA256; no original notebook was executed, no checkpoint deserialized.')
audit["source_hashes_after"]=after
audit["source_preservation_pass"]=preserved
figure_presence={name:(DIAG/name).is_file() for name in ["feature_correlation_drift.png","temporal_distribution_shift.png"]}
claim("Phase 2C provenance","Existing diagnostic figures","PARTIAL",f"Presence: {figure_presence}. Underlying JSON reproduced; pixel-to-data correspondence and generation code are not independently recoverable from the missing notebook.")
audit["phase2c_figure_presence"]=figure_presence
audit["claims"]=findings
audit["scientific_reference"]={
    "title":"Mignan & Woessner (2012), Estimating the magnitude of completeness for earthquake catalogs",
    "url":"https://www.corssa.org/export/sites/corssa/.galleries/articles-pdf/Mignan-Woessner-2012-CORSSA-Magnitude-of-completeness.pdf_2063069299.pdf",
    "relevant_sections":"4-7: definition, fitted catalog/network methods, uncertainty, and temporal/spatial variability; section5.1 frequency-magnitude relation",
    "use":"Definition/methodological limits only; no Mc estimate made for this dataset."
}
audit["verdicts"]={
    "phase2b_core_numerics":"PASS",
    "phase2b_report_accuracy":"FAIL",
    "recorded_control_evidence":"PASS",
    "exact_historical_control_equivalence":"UNKNOWN",
    "validation_only_source":"PASS",
    "absolute_historical_no_test_access":"UNKNOWN",
    "phase2c_json_correlations":"PASS",
    "phase2c_method_provenance":"PARTIAL",
    "phase2c_report_feature_statistics":"FAIL",
    "descriptive_distribution_shift":"PASS",
    "coverage_adjusted_event_frequency_and_inferential_significance":"PARTIAL",
    "catalog_completeness_values":"UNKNOWN",
    "causal_catalog_completeness_claim":"FAIL",
    "registered_artifact_integrity":"PASS",
    "phase2c_no_training_claim":"UNKNOWN"
}
savej("verification_metrics.json",audit)
pd.DataFrame(findings).to_csv(OUT/"discrepancy_table.csv",index=False)
feature_frame.to_csv(OUT/"feature_verification.csv",index=False)
def markdown_table(headers,records):
    return "\n".join(["| "+" | ".join(headers)+" |","|"+"|".join(["---"]*len(headers))+"|"]+
                     ["| "+" | ".join(str(v) for v in row)+" |" for row in records])
b_table=markdown_table(["Validation metric","delta1","delta2"],[
    ("MAE",f'{a["mae"]:.6f}',f'{b["mae"]:.6f}'),
    ("RMSE",f'{a["rmse"]:.6f}',f'{b["rmse"]:.6f}'),
    ("R2",f'{a["r2"]:.6f}',f'{b["r2"]:.6f}'),
    ("Prediction SD (population)",f'{a["pred_std"]:.6f}',f'{b["pred_std"]:.6f}'),
    ("ML>=4.5 MAE; n=75",f'{a["high_mae"]:.6f}',f'{b["high_mae"]:.6f}'),
    ("ML>=4.5 underprediction","75/75","75/75"),
    (f'ML<3 MAE; n={a["low_count"]}',f'{a["low_mae"]:.6f}',f'{b["low_mae"]:.6f}'),
    ("Clean train MAE (history)",f'{a["clean_train_mae_logged"]:.6f}',f'{b["clean_train_mae_logged"]:.6f}'),
    ("Val-train MAE gap (history)",f'{a["gap_val_minus_train_mae_logged"]:.6f}',f'{b["gap_val_minus_train_mae_logged"]:.6f}'),
    ("Best/total epochs",f'{a["best_epoch"]}/{a["epochs_run"]}',f'{b["best_epoch"]}/{b["epochs_run"]}'),
    ("Recorded training seconds",f'{a["training_seconds_logged"]:.6f}',f'{b["training_seconds_logged"]:.6f}')
])
f_table=markdown_table(["Feature","Pearson H1","Pearson H2","Pearson val","Spearman train","Spearman val"],[
    (r["name"],f'{r["r_h1"]:.6f}',f'{r["r_h2"]:.6f}',f'{r["r_val"]:.6f}',
     f'{r["rho_train"]:.6f}',f'{r["rho_val"]:.6f}') for r in rows])
t_table=markdown_table(["Split","n","Target mean","ML>=4.5","Elapsed years","N/elapsed year","Mean 30d rate"],[
    (s,v["n"],f'{v["target_mean"]:.6f}',f'{v["high_count"]} ({v["high_percent"]:.3f}%)',
     f'{v["elapsed_years_365_2425"]:.6f}',f'{v["targets_per_elapsed_year_N_over_span"]:.3f}',
     f'{v["last_input_rate_30d_mean"]:.6f}') for s,v in summaries.items()])
h1,h2=periods["train_h1"],periods["train_h2"]
report=f"""# Independent verification of Gemini Phase 2B and 2C

Audit date: 2026-11-10. Verification only: no training, tuning, model inference, data regeneration or model selection. Original model checkpoints were hashed as bytes, never deserialized. Test targets were accessed only to reproduce Phase 2C descriptive statistics.

PASS means independently reproduced; PARTIAL means supported with limitations; FAIL means contradicted; UNKNOWN means insufficient evidence.

## Verdict

**PASS:** Phase 2B core validation metrics; recorded matching architecture, initialization and sampling; Phase 2C JSON feature correlations and descriptive distribution shift; registered artifact integrity.
**FAIL:** Several narrative numbers and interpretations in both reports.
**UNKNOWN:** Exact historical one-variable control equivalence, absolute absence of historical test access, formal Mc values and whether Phase 2C executed any unrecorded training.
**PARTIAL:** Reconstructed Phase 2C method provenance, inferential significance and coverage-adjusted event-frequency interpretation.

## Phase 2B: independently recalculated validation results

Both saved prediction files' targets exactly match the corrected-v2 validation labels in manifest order (n=1065). The original registered protocol specifies Huber delta1=1.0; the new notebook specifies delta2=2.0. Identity target scaling makes both thresholds original ML units.

{b_table}

MAE improves by {changes["mae_delta2_minus_delta1"]:.6f} ML ({changes["mae_improvement_percent"]:.4f}%), while prediction SD falls {changes["prediction_sd_reduction_percent"]:.4f}% and high-magnitude MAE worsens {changes["high_mae_change"]:.6f} ML. This is not evidence of a useful generalization breakthrough. The smaller logged train/validation gap accompanies worse training MAE and greater compression; it does not establish reduced overfitting. Paired absolute error improves on {changes["paired_error_wins"]} and worsens on {changes["paired_error_losses"]} validation examples. Statistical reliability remains uncertain: one seed, overlapping windows, 75 high targets, and checkpoint selection on this same validation set. No new significance experiment was introduced.

**Report corrections:** delta2 Huber loss is **{b["huber"]:.6f}**, not 0.2312. Its prediction mean moves **farther** from the actual mean (absolute offset {a["mean_distance_from_actual"]:.6f} to {b["mean_distance_from_actual"]:.6f}). The {100*b["predicted_actual_sd_ratio"]:.3f}% SD ratio corresponds to **{100*b["predicted_actual_variance_ratio"]:.3f}% of target variance**, not 19.8%. Recorded runtime is {b["training_seconds_logged"]:.6f}s, not 10.3s. Objective losses with different deltas are not directly comparable quality scores. The RMSE plotting expression is fragile but uses the correct RMSE fallback for the saved baseline history, which has no val_pred_std field.

### Controls and provenance limits

The architecture AST is identical; recorded initial weight hashes match; all 19 common epochs have identical sampled-parent/order hashes and counts. The approved augmentation consumer source hash matches the original protocol. The new code reads the registered full-precision settings: hidden128, layers2, dropout {hyper["dropout"]}, learning rate {hyper["learning_rate"]}, weight decay {hyper["weight_decay"]}, batch128; AdamW over all parameters, seed42, max50, patience8 and clipping1.0. There is no warm-start load in the supplied new training cell.

The saved best-checkpoint hashes match their fit summaries. Recorded minimum-validation-MAE epochs12/11 agree with saved predictions; source saves on validation improvement and reloads best before generating those predictions. Weight-to-prediction identity was not rerun.

The delta2 source indexes training/validation tensors only and selects using validation MAE. **Exact historical control equivalence and an absolute no-test-access guarantee remain UNKNOWN:** no immutable delta2 configuration/environment snapshot or independent access/execution log establishes every runtime fact. File hashing/memory mapping of the monolithic dataset is not evidence that test labels influenced optimization. The separate huber_experiment folder preserves Phase 2 outputs, but rerunning its notebook would overwrite its own artifacts; no safe resume logic is supplied.

## Phase 2C: feature verification

All **{len(feature_checks)} numeric JSON fields** reproduce, using inverse-scaled **final historical step19**, not whole-sequence averages. Feature names/indexes match the corrected schema; labels match source IDs; targets are unique across splits; timestamps are sorted; every input precedes its target.

{f_table}

Matching early/late reconstruction: H1 rows[0,{half}), n={h1["n"]}, targets {h1["first_target"]} through {h1["last_target"]}; H2 rows[{half},4973), n={h2["n"]}, targets {h2["first_target"]} through {h2["last_target"]}. These reproduce all H1/H2 correlations. **The named phase2c_feature_temporal_audit.ipynb is absent in the scoped project search.** The boundary definition is independently reconstructed, not recovered original source.

The report's feature table has **{sum(not x["match"] for x in table_checks)}/{len(table_checks)} incorrect mean/SD entries** beyond displayed rounding tolerance; corrected values are in feature_verification.csv. Examples: validation mean_magnitude_20={rows[6]["val_mean"]:.6f}+/-{rows[6]["val_std"]:.6f}, historical_depth_km={rows[1]["val_mean"]:.6f}+/-{rows[1]["val_std"]:.6f}, and training inter_event_days SD={rows[4]["tr_std"]:.6f}. Depth redundancy is pair8/9, r={corrmatrix[8,9]:.6f}, not pair1/8. Reported Cohen d uses overall-split SD, not pooled within-group SD. The claim that no validation |r| exceeds0.20 is contradicted by rolling-mean r={rows[6]["r_val"]:.6f}. Individual correlations do not establish absence of nonlinear/joint/sequence predictive signal or joint held-out explained variance.

## Temporal statistics and scientific interpretation

{t_table}

Rates in the table are retained-ML events/day; mean 30-day test/train ratio={audit["phase2c_temporal"]["mean_final_step_rate_test_train_ratio"]:.4f}. Final-input gaps and rolling rates were recomputed from explicit UTC nanoseconds with interval (time-30days,time]; target gaps independently match the manifest. Final-input gap medians are {summaries["train"]["last_input_inter_event_days_median"]:.6f}/{summaries["val"]["last_input_inter_event_days_median"]:.6f}/{summaries["test"]["last_input_inter_event_days_median"]:.6f} days.

Train-to-validation Wasserstein={distances["val"]["wasserstein"]:.6f}, KS D={distances["val"]["ks_statistic"]:.6f}; train-to-test Wasserstein={distances["test"]["wasserstein"]:.6f}. The nominal validation KS p={distances["val"]["ks_nominal_pvalue"]:.4g} is reproducible, but it is not a dependence-robust significance claim. Overlapping histories, time trends and tied magnitudes limit iid Pearson/Spearman/KS p-value interpretations. Saved p=0 values reflect numerical underflow.

Annual rates above use exact first/last target elapsed seconds divided by365.2425days, handling partial years; both N/span and (N-1)/span are saved. Gemini's rounded 1.13/0.67-year denominators introduce approximation error. These are retained ML target counts, **not complete regional seismicity rates**; catalog uptime, gaps, magnitude-selection/reporting policies and true observation exposure are unknown.

**Confirmed:** empirical temporal shifts, correlation weakening/sign changes, and compressed validation predictions.
**Hypotheses:** changing detection/reporting thresholds, spatial sampling or physical clustering contributing to those shifts. No validated Mc fit, uncertainty estimate, station-coverage history, learned-rule analysis or conditional-independence test was supplied. Historical Mc~4.0 and modern Mc~1.5 remain **UNKNOWN**; causal completeness and forced-collapse explanations are unsupported as established conclusions.

Mc needs a defined estimation/validation procedure, rather than an observed minimum. The Gutenberg-Richter relation describes a frequency-magnitude distribution and does not specify temporal independence. See [Mignan & Woessner (2012), sections4-7]({audit["scientific_reference"]["url"]}). This methodological reference does not establish completeness or any cause for this catalog.

## Integrity and required corrections

All **{len(integrity)} registered original Phase 2/v2 hashes match**, including four original best checkpoints, corrected .pt/.npz, schema, manifest and scalers. All {len(snapshot)} inspected source hashes remain identical before/after this audit. Registered model preservation is verified; the missing Phase 2C execution source prevents certifying that no other training occurred.

Before another research experiment: correct the separately enumerated report numbers/labels, recover the original Phase 2C notebook and execution provenance, preserve a complete immutable Phase 2B run configuration, and downgrade completeness/independence/causation assertions to untested hypotheses. Preserve this verification and the original reports side by side; nothing was silently repaired.

Evidence: [verification_metrics.json](verification_metrics.json), [discrepancy_table.csv](discrepancy_table.csv), [feature_verification.csv](feature_verification.csv), and [executed verification notebook](../../phase2d_gemini_verification.ipynb). Gemini originals: ../huber_experiment/huber_experiment_report.md and ../error_diagnosis/phase2c_feature_informativeness_report.md. Verification stops here.
"""
(OUT/"verification_report.md").write_text(report,encoding="utf-8")
marker="## Phase 2D independent Gemini verification — 2026-11-10"
note=f"""{marker}

- Verification only completed; no model loading, training, tuning, preprocessing or selection.
- PASS: Phase2B core validation metrics; matching recorded initialization and all19 common sampler epochs; all340 Phase2C JSON fields; all29 registered original Phase2/v2 hashes.
- FAIL: delta2 Huber/runtime/mean-shift/variance prose and19 Phase2C feature-table summaries; unsupported causal completeness/independence conclusions.
- UNKNOWN: exact historical control equivalence, absolute historical test non-access, Mc values, missing Phase2C notebook/execution provenance.
- Audit: phase2d_gemini_verification.ipynb; phase2_bilstm_results/gemini_verification/verification_report.md. Existing reports retained; no subsequent experiment started.

"""
for path in [ROOT/"project_status.md",ROOT/".agent-memory/checkpoint.md"]:
    old=path.read_text(encoding="utf-8") if path.exists() else ""
    if marker not in old:
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_text(note+old,encoding="utf-8")
print("Saved independent report, metrics, discrepancy/feature tables; source preservation PASS.")
print("Verdicts:",json.dumps(audit["verdicts"],sort_keys=True))


Saved independent report, metrics, discrepancy/feature tables; source preservation PASS.
Verdicts: {"absolute_historical_no_test_access": "UNKNOWN", "catalog_completeness_values": "UNKNOWN", "causal_catalog_completeness_claim": "FAIL", "coverage_adjusted_event_frequency_and_inferential_significance": "PARTIAL", "descriptive_distribution_shift": "PASS", "exact_historical_control_equivalence": "UNKNOWN", "phase2b_core_numerics": "PASS", "phase2b_report_accuracy": "FAIL", "phase2c_json_correlations": "PASS", "phase2c_method_provenance": "PARTIAL", "phase2c_no_training_claim": "UNKNOWN", "phase2c_report_feature_statistics": "FAIL", "recorded_control_evidence": "PASS", "registered_artifact_integrity": "PASS", "validation_only_source": "PASS"}
